# AF3 vs Design Structure Alignment (TM-score)

This notebook compares each design structure against its matching AF3 re-prediction in:
- `AF3_Outputs/<model>_<condition>_run_XXX_cycle_YY/<...>_model.cif`

Outputs:
- Per-structure TM-score table
- Summary by model / condition / cycle
- Quick plots


In [ ]:
# Optional, only if imports fail:
# %pip install numpy pandas matplotlib biopython


## Tool 1: Structure alignment + TM-score

This is the primary tool cell. It:
1. Resolves each AF3 output to its source design CIF
2. Aligns CA atoms using Kabsch superposition
3. Computes TM-score (normalized by design length and AF3 length)


In [ ]:
from __future__ import annotations

import csv
import re
from pathlib import Path
from dataclasses import dataclass
from typing import Optional
from statistics import mean, median

import numpy as np

from Bio.PDB import MMCIFParser, is_aa
from Bio.PDB.Polypeptide import protein_letters_3to1
from Bio.Align import PairwiseAligner

try:
    import pandas as pd
    HAS_PANDAS = True
except Exception:
    HAS_PANDAS = False

CASE_RE = re.compile(r"^(boltz|intellifold|openfold3)_(.+)_run_(\d{3})_cycle_(\d{2})$")


@dataclass
class MatchCase:
    model: str
    condition: str
    run: str
    cycle: str


def parse_case_name(case_name: str) -> Optional[MatchCase]:
    m = CASE_RE.match(case_name)
    if not m:
        return None
    return MatchCase(model=m.group(1), condition=m.group(2), run=m.group(3), cycle=m.group(4))


def pick_af3_model_cif(case_dir: Path) -> Optional[Path]:
    expected = case_dir / f"{case_dir.name}_model.cif"
    if expected.exists():
        return expected
    candidates = sorted(case_dir.glob("*_model.cif"))
    return candidates[0] if candidates else None


def find_design_cif(campaign_root: Path, m: MatchCase) -> Optional[Path]:
    base = campaign_root / m.model / m.condition / f"run_{m.run}" / f"cycle_{m.cycle}"
    candidates = [
        base / "pred_min" / "model_0.cif",
        base / "pred_min" / "model_0_UNKPATCH.cif",
    ]
    for c in candidates:
        if c.exists():
            return c
    if m.model == "openfold3":
        seed_models = sorted((base / "openfold3").glob("*/seed_*/*_model.cif"))
        if seed_models:
            return seed_models[0]
    return None


def _resname_to_aa(resname: str) -> str:
    key = resname.strip().upper()
    if len(key) == 3 and key in protein_letters_3to1:
        return protein_letters_3to1[key]
    return "X"


def extract_ca_seq_coords(cif_path: Path, chain_id: Optional[str] = None):
    parser = MMCIFParser(QUIET=True)
    structure = parser.get_structure(cif_path.stem, str(cif_path))
    model = next(structure.get_models())
    chains = list(model.get_chains())
    if not chains:
        raise ValueError(f"No chains found in {cif_path}")

    if chain_id is None:
        best_chain = None
        best_count = -1
        for ch in chains:
            count = 0
            for res in ch:
                if is_aa(res, standard=False) and "CA" in res:
                    count += 1
            if count > best_count:
                best_count = count
                best_chain = ch
        chain = best_chain
    else:
        chain = model[chain_id]

    seq = []
    coords = []
    for res in chain:
        if not is_aa(res, standard=False):
            continue
        if "CA" not in res:
            continue
        seq.append(_resname_to_aa(res.get_resname()))
        coords.append(res["CA"].coord.astype(float))

    if not coords:
        raise ValueError(f"No protein CA atoms found in {cif_path}")

    return "".join(seq), np.asarray(coords, dtype=float)


def map_aligned_indices(seq_a: str, seq_b: str):
    aligner = PairwiseAligner()
    aligner.mode = "global"
    aligner.match_score = 1.0
    aligner.mismatch_score = 0.0
    aligner.open_gap_score = -1.0
    aligner.extend_gap_score = -0.5

    aln = aligner.align(seq_a, seq_b)[0]
    idx_a = []
    idx_b = []
    for (a0, a1), (b0, b1) in zip(aln.aligned[0], aln.aligned[1]):
        block_len = min(a1 - a0, b1 - b0)
        for k in range(block_len):
            idx_a.append(a0 + k)
            idx_b.append(b0 + k)

    if not idx_a:
        raise ValueError("No aligned residue positions found")

    return np.asarray(idx_a, dtype=int), np.asarray(idx_b, dtype=int)


def kabsch_superpose(mobile: np.ndarray, target: np.ndarray):
    mob_cent = mobile.mean(axis=0)
    tar_cent = target.mean(axis=0)
    mob0 = mobile - mob_cent
    tar0 = target - tar_cent

    cov = mob0.T @ tar0
    v, _, wt = np.linalg.svd(cov)
    det = np.linalg.det(v @ wt)
    d = np.diag([1.0, 1.0, np.sign(det)])
    rot = v @ d @ wt
    mobile_aligned = mob0 @ rot + tar_cent
    return mobile_aligned


def tm_score_from_distances(distances: np.ndarray, norm_len: int) -> float:
    if norm_len <= 0:
        return float("nan")
    if norm_len > 15:
        d0 = 1.24 * ((norm_len - 15) ** (1.0 / 3.0)) - 1.8
        d0 = max(d0, 0.5)
    else:
        d0 = 0.5
    return float(np.sum(1.0 / (1.0 + (distances / d0) ** 2)) / norm_len)


def align_and_score(design_cif: Path, af3_cif: Path, chain_id: Optional[str] = None):
    seq_d, xyz_d = extract_ca_seq_coords(design_cif, chain_id=chain_id)
    seq_a, xyz_a = extract_ca_seq_coords(af3_cif, chain_id=chain_id)

    idx_d, idx_a = map_aligned_indices(seq_d, seq_a)
    d_sel = xyz_d[idx_d]
    a_sel = xyz_a[idx_a]

    d_aligned = kabsch_superpose(d_sel, a_sel)
    distances = np.linalg.norm(d_aligned - a_sel, axis=1)

    rmsd = float(np.sqrt(np.mean(distances ** 2)))
    tm_design_norm = tm_score_from_distances(distances, len(seq_d))
    tm_af3_norm = tm_score_from_distances(distances, len(seq_a))
    tm_mean = float(np.nanmean([tm_design_norm, tm_af3_norm]))

    return {
        "aligned_ca": int(len(distances)),
        "design_len": int(len(seq_d)),
        "af3_len": int(len(seq_a)),
        "rmsd_ca": rmsd,
        "tm_design_norm": tm_design_norm,
        "tm_af3_norm": tm_af3_norm,
        "tm_mean": tm_mean,
    }


def run_af3_design_tmscore(campaign_root: Path, chain_id: Optional[str] = None):
    af3_root = campaign_root / "AF3_Outputs"
    if not af3_root.exists():
        raise FileNotFoundError(f"AF3 output folder not found: {af3_root}")

    rows = []
    missing = []

    for case_dir in sorted(p for p in af3_root.iterdir() if p.is_dir()):
        case_name = case_dir.name
        parsed = parse_case_name(case_name)
        if parsed is None:
            missing.append({"case_name": case_name, "reason": "name_parse_failed"})
            continue

        af3_cif = pick_af3_model_cif(case_dir)
        if af3_cif is None:
            missing.append({"case_name": case_name, "reason": "af3_model_cif_missing"})
            continue

        design_cif = find_design_cif(campaign_root, parsed)
        if design_cif is None:
            missing.append({"case_name": case_name, "reason": "design_cif_missing"})
            continue

        try:
            score = align_and_score(design_cif, af3_cif, chain_id=chain_id)
            rows.append({
                "case_name": case_name,
                "model": parsed.model,
                "condition": parsed.condition,
                "run": parsed.run,
                "cycle": parsed.cycle,
                "design_cif": str(design_cif),
                "af3_cif": str(af3_cif),
                **score,
            })
        except Exception as exc:
            missing.append({"case_name": case_name, "reason": f"alignment_failed: {exc}"})

    return rows, missing


In [ ]:
# Configure root folder and run tool 1
CAMPAIGN_ROOT = Path('/Users/thomasfryer/iProteinHunter/output/test_helixkill_6cond_12runs_allpredictors_20260226_181547')
CHAIN_ID = None  # For single-chain monomer outputs, keep None.

rows, missing = run_af3_design_tmscore(CAMPAIGN_ROOT, chain_id=CHAIN_ID)
print(f"Scored pairs: {len(rows)}")
print(f"Missing/failed: {len(missing)}")

if HAS_PANDAS:
    df_scores = pd.DataFrame(rows)
    df_missing = pd.DataFrame(missing)
    display(df_scores.head())
else:
    print('pandas not installed; showing first 5 rows as dicts:')
    for r in rows[:5]:
        print(r)


In [ ]:
# Save outputs
out_scores = CAMPAIGN_ROOT / 'af3_tmscore_per_structure.csv'
out_missing = CAMPAIGN_ROOT / 'af3_tmscore_missing.csv'

score_fields = [
    'case_name', 'model', 'condition', 'run', 'cycle',
    'design_cif', 'af3_cif',
    'aligned_ca', 'design_len', 'af3_len',
    'rmsd_ca', 'tm_design_norm', 'tm_af3_norm', 'tm_mean',
]

with out_scores.open('w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=score_fields)
    w.writeheader()
    for r in sorted(rows, key=lambda x: (x['model'], x['condition'], x['run'], x['cycle'])):
        w.writerow(r)

with out_missing.open('w', newline='') as f:
    fields = ['case_name', 'reason']
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    for r in missing:
        w.writerow(r)

print(f'Wrote: {out_scores}')
print(f'Wrote: {out_missing}')


In [ ]:
# Summary
if HAS_PANDAS:
    df_scores = pd.DataFrame(rows)
    summary_model = (
        df_scores.groupby('model')[['tm_mean', 'tm_design_norm', 'tm_af3_norm', 'rmsd_ca']]
        .agg(['mean', 'median', 'std'])
    )
    summary_condition = (
        df_scores.groupby(['model', 'condition'])[['tm_mean', 'rmsd_ca']]
        .agg(['mean', 'median', 'std'])
    )
    display(summary_model)
    display(summary_condition.head(30))
else:
    by_model = {}
    for r in rows:
        by_model.setdefault(r['model'], {'tm': [], 'rmsd': []})
        by_model[r['model']]['tm'].append(r['tm_mean'])
        by_model[r['model']]['rmsd'].append(r['rmsd_ca'])
    for model, vals in sorted(by_model.items()):
        print(model)
        print('  tm_mean mean/median:', round(mean(vals['tm']), 4), round(median(vals['tm']), 4))
        print('  rmsd mean/median   :', round(mean(vals['rmsd']), 4), round(median(vals['rmsd']), 4))


In [ ]:
# Optional plots
try:
    import matplotlib.pyplot as plt
except Exception:
    print('matplotlib not installed; skip plots or run: %pip install matplotlib')
    raise

if HAS_PANDAS:
    df_scores = pd.DataFrame(rows)

    plt.figure(figsize=(10, 4))
    for model, sub in sorted(df_scores.groupby('model')):
        vals = sub['tm_mean'].sort_values().values
        plt.plot(vals, label=model, linewidth=1.5)
    plt.ylabel('TM-score (mean norm)')
    plt.xlabel('Structure pairs (sorted within model)')
    plt.title('AF3 vs design structural agreement')
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(9, 4))
    df_scores.boxplot(column='tm_mean', by=['model'], grid=False)
    plt.suptitle('')
    plt.title('TM-score by model')
    plt.ylabel('TM-score (mean norm)')
    plt.tight_layout()
    plt.show()
else:
    print('Install pandas for plotting from grouped tables: %pip install pandas matplotlib')


## Notes

- `tm_mean` is the average of TM-scores normalized by design length and AF3 length.
- `rmsd_ca` is CA RMSD after Kabsch superposition on aligned residues.
- If you need strict chain control for multichain data, set `CHAIN_ID` explicitly.
